In [1]:
import pandas as pd
import re

# 1. 데이터 불러오기
df_trade = pd.read_csv('아파트실거래가_2021_2025.csv', encoding='utf-8-sig', low_memory=False)
df_kapt = pd.read_excel('20260522_단지_기본정보.xlsx', skiprows=1)

# 2. 마스터키 정제
df_trade['조인용_주소'] = df_trade['시군구'].astype(str).str.strip() + ' ' + df_trade['번지'].astype(str).str.strip()

def clean_kapt_address(row):
    addr = str(row['법정동주소']).strip()
    apt = str(row['단지명']).strip()
    if apt in addr:
        addr = addr.replace(apt, '').strip()
    return addr

df_kapt['조인용_주소'] = df_kapt.apply(clean_kapt_address, axis=1)

df_trade['조인용_주소'] = df_trade['조인용_주소'].str.replace(r'\s+', ' ', regex=True)
df_kapt['조인용_주소'] = df_kapt['조인용_주소'].str.replace(r'\s+', ' ', regex=True)

df_kapt = df_kapt.drop_duplicates(subset=['조인용_주소'], keep='first')

# 3. 데이터 병합
df_main = pd.merge(df_trade, df_kapt, on='조인용_주소', how='left')

if '단지명_x' in df_main.columns:
    df_main = df_main.rename(columns={'단지명_x': '단지명'})
    df_main = df_main.drop(columns=['단지명_y'], errors='ignore')

# 4. 결과 확인 및 저장
print(f"원본 실거래가 데이터 개수: {len(df_trade):,}건")
print(f"병합 후 메인 데이터 개수: {len(df_main):,}건")

unmatched_count = df_main['동수'].isna().sum()
matched_count = len(df_main) - unmatched_count
print(f"매칭된 데이터 개수: {matched_count:,}건")
print(f"누락된 데이터 개수: {unmatched_count:,}건")
df_main.to_csv('main_data.csv', index=False, encoding='utf-8-sig')

원본 실거래가 데이터 개수: 12,454건
병합 후 메인 데이터 개수: 12,454건
매칭된 데이터 개수: 10,106건
누락된 데이터 개수: 2,348건


In [2]:
import requests

# 1. 데이터 불러오기 및 검색용 주소 자동 인식
df_main = pd.read_csv('main_data.csv', encoding='utf-8-sig', low_memory=False)

if '조인용_주소' in df_main.columns:
    df_main['검색용_주소'] = df_main['조인용_주소']
elif '시군구' in df_main.columns and '번지' in df_main.columns:
    df_main['검색용_주소'] = df_main['시군구'].astype(str).str.strip() + ' ' + df_main['번지'].astype(str).str.strip()

# 2. 고유 주소만 추출
unique_addresses = pd.DataFrame(df_main['검색용_주소'].unique(), columns=['검색용_주소'])
print(f"카카오 API에 검색할 고유 아파트 주소 개수: {len(unique_addresses)}개")

# 3. 카카오 로컬 API 지오코딩 함수
REST_API_KEY = "02ce952b5588d51d2fb6034d03ae55dd"

def get_lat_lon(address):
    url = f"https://dapi.kakao.com/v2/local/search/address.json?query={address}"
    headers = {"Authorization": f"KakaoAK {REST_API_KEY}"}
    
    try:
        response = requests.get(url, headers=headers)
        result = response.json()
        
        if result['documents']: 
            match = result['documents'][0]
            # y가 위도, x가 경도
            return pd.Series([float(match['y']), float(match['x'])])
        else:
            return pd.Series([None, None])
            
    except Exception as e:
        return pd.Series([None, None])
# 4. API 호출 및 위경도 추출 
unique_addresses[['아파트위도', '아파트경도']] = unique_addresses['검색용_주소'].apply(get_lat_lon)

# 5. 원본 데이터에 위경도 병합 및 저장
df_final = pd.merge(df_main, unique_addresses, on='검색용_주소', how='left')

# 임시 컬럼 삭제
df_final = df_final.drop(columns=['검색용_주소'])

missing_geo = df_final['아파트위도'].isna().sum()
print(f"원본 데이터 수: {len(df_main):,}건 / 병합 후 데이터 수: {len(df_final):,}건")
print(f"누락된 데이터: {missing_geo:,}건")

# 최종 저장
df_final.to_csv('main_data_with_geo.csv', index=False, encoding='utf-8-sig')

카카오 API에 검색할 고유 아파트 주소 개수: 181개
원본 데이터 수: 12,454건 / 병합 후 데이터 수: 12,454건
누락된 데이터: 0건


In [3]:
import numpy as np

# 1. 원본 메인 데이터 로드
df = pd.read_csv('main_data_with_geo.csv', encoding='utf-8-sig', low_memory=False)

# 2. 외부 공간 데이터 로드
df_subway = pd.read_csv('subway_geo_seongbuk.csv', encoding='utf-8-sig').dropna(subset=['역위도', '역경도'])
df_park = pd.read_csv('성북구_공원.csv', encoding='utf-8-sig')
df_hosp = pd.read_csv('성북구_병원.csv', encoding='utf-8-sig')

df_bus = pd.read_excel('서울시 버스정류소 위치정보.xlsx')
df_bus = df_bus.drop_duplicates(subset=['X좌표', 'Y좌표']) 

df_smallbiz = pd.read_csv('성북구_소상공인.csv', encoding='utf-8-sig')

biz_col = '상권업종중분류명' 

if biz_col in df_smallbiz.columns:
    keywords = '소매|커피|카페|약국|제과|베이커리|편의점|슈퍼|마트'
    df_smallbiz = df_smallbiz[df_smallbiz[biz_col].str.contains(keywords, na=False, regex=True)]

# 3. 하버사인 거리 계산 및 공간 변수 생성 함수
def calculate_haversine(lat1, lon1, lat2_array, lon2_array):
    R = 6371000.0
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2_array, lon2_array = np.radians(lat2_array), np.radians(lon2_array)
    dlat = lat2_array - lat1
    dlon = lon2_array - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2_array) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def add_geo_features(row):
    lat, lon = row['아파트위도'], row['아파트경도']
    
    if pd.isna(lat) or pd.isna(lon):
        return pd.Series({'역세권여부': 0, '반경500m_버스정류장': 0, '500m내_편의시설_총합': 0})
    
    dist_sub = calculate_haversine(lat, lon, df_subway['역위도'].values, df_subway['역경도'].values)
    is_station = 1 if len(dist_sub) > 0 and np.min(dist_sub) <= 500 else 0
    
    dist_bus = calculate_haversine(lat, lon, df_bus['Y좌표'].values, df_bus['X좌표'].values)
    bus_count = np.sum(dist_bus <= 500)
    
    park_cnt = np.sum(calculate_haversine(lat, lon, df_park['위도'].values, df_park['경도'].values) <= 500)
    hosp_cnt = np.sum(calculate_haversine(lat, lon, df_hosp['위도'].values, df_hosp['경도'].values) <= 500)
    smallbiz_cnt = np.sum(calculate_haversine(lat, lon, df_smallbiz['위도'].values, df_smallbiz['경도'].values) <= 500)
    
    return pd.Series({
        '역세권여부': is_station,
        '반경500m_버스정류장': int(bus_count),
        '500m내_편의시설_총합': int(park_cnt + hosp_cnt + smallbiz_cnt)
    })

# 4. 데이터 병합 및 통째로 저장
geo_cols = df.apply(add_geo_features, axis=1)

df_final = pd.concat([df, geo_cols], axis=1)

df_final.to_csv('final_dataset.csv', index=False, encoding='utf-8-sig')

print(f"최종 컬럼 개수: 기존 {len(df.columns)}개 -> 추가 후 {len(df_final.columns)}개")

최종 컬럼 개수: 기존 107개 -> 추가 후 110개


In [4]:
print(" 파생변수 전처리 및 데이터셋 업데이트를 시작합니다...")

# 1. 원본 데이터 불러오기
df = pd.read_csv('final_dataset.csv', low_memory=False)
initial_col_count = len(df.columns)

# 2. 파생변수 전처리 및 추가
if '전용면적(㎡)' in df.columns:
    df['전용면적'] = df['전용면적(㎡)']

if '계약년월' in df.columns and '건축년도' in df.columns:
    df['건물나이'] = df['계약년월'].astype(str).str[:4].astype(int) - df['건축년도']

if '총주차대수' in df.columns and '세대수' in df.columns:
    df['세대당_주차대수'] = df['총주차대수'] / df['세대수'].replace(0, np.nan)
    df['세대당_주차대수'] = df['세대당_주차대수'].fillna(0) 

# 3. 분석 편의를 위한 종속변수 정제
if '거래금액(만원)' in df.columns:
    if df['거래금액(만원)'].dtype == 'object':
        df['거래금액_만원'] = df['거래금액(만원)'].astype(str).str.replace(',', '').astype(float)
    else:
        df['거래금액_만원'] = df['거래금액(만원)']

# 4. 기존 파일에 업데이트
df.to_csv('final_dataset.csv', index=False, encoding='utf-8-sig')

# 5. 결과 확인 및 최종 컬럼 개수 출력

added_cols = ['전용면적', '건물나이', '세대당_주차대수', '거래금액_만원']
print("\n[추가된 변수 미리보기]")
print(df[added_cols].head())

# 🔥 요청하신 최종 컬럼 개수 브리핑
final_col_count = len(df.columns)
print("-" * 50)
print(f"원본 컬럼 개수: {initial_col_count}개")
print(f"업데이트 후 최종 컬럼 개수: {final_col_count}개 (+{final_col_count - initial_col_count}개 추가됨)")
print("-" * 50)

 파생변수 전처리 및 데이터셋 업데이트를 시작합니다...

[추가된 변수 미리보기]
      전용면적  건물나이  세대당_주차대수  거래금액_만원
0  59.8800    16  1.198376  60000.0
1  59.9400    22  0.981693  65000.0
2  15.2144     2  0.511706  11550.0
3  84.9600    21  0.871203  76000.0
4  59.5800    23  1.222703  71000.0
--------------------------------------------------
원본 컬럼 개수: 110개
업데이트 후 최종 컬럼 개수: 114개 (+4개 추가됨)
--------------------------------------------------


In [5]:
print(" '기준금리', '층(고층여부)', '브랜드_Top20' 전처리 및 업데이트를 시작합니다...")

# 1. 파일 불러오기
df = pd.read_csv('final_dataset.csv', low_memory=False)
initial_col_count = len(df.columns)

# 2. 파생변수 전처리 및 추가
if '계약년월' in df.columns:
    df['계약년'] = df['계약년월'].astype(str).str[:4].astype(int)
    
    try:
        df_interest = pd.read_csv('interest_clean.csv')
        # 계약년을 기준으로 병합 (Left Join)
        df = pd.merge(df, df_interest, on='계약년', how='left')
        # 빈칸이 생기면 이전 데이터로 채우거나 0으로 처리
        df['기준금리'] = df['기준금리'].fillna(method='ffill').fillna(0)
    except FileNotFoundError:
        print("'interest_clean.csv' 파일이 없어 기준금리를 0으로 임시 채웁니다.")
        df['기준금리'] = 0

if '층' in df.columns:
    
    df['층'] = pd.to_numeric(df['층'], errors='coerce').fillna(0)
    
    df['고층여부'] = (df['층'] >= 15).astype(int)

top_brands = '래미안|자이|푸르지오|이편한|e편한|힐스테이트|더샵|롯데캐슬|아이파크|SK|데시앙|센트레빌|포레나|꿈에그린|스위첸|어울림|리슈빌|하늘채|베르디움|호반|우미'
brand_mask = pd.Series(False, index=df.index)

if '시공사' in df.columns:
    brand_mask = brand_mask | df['시공사'].astype(str).str.contains(top_brands, na=False, regex=True)
if '단지명' in df.columns:
    brand_mask = brand_mask | df['단지명'].astype(str).str.contains(top_brands, na=False, regex=True)

df['브랜드_Top20_여부'] = brand_mask.astype(int)

# 3. 기존 파일에 업데이트
df.to_csv('final_dataset.csv', index=False, encoding='utf-8-sig')

# 4. 결과 확인 및 최종 컬럼 개수 출력
print("\n 변수들이 업데이트 되었습니다")

added_cols = ['기준금리', '층', '고층여부', '브랜드_Top20_여부']

if '계약년' in df.columns:
    added_cols.insert(0, '계약년')

print("\n[새로 업데이트된 변수 미리보기]")
print(df[added_cols].head())

final_col_count = len(df.columns)
print("-" * 50)
print(f"이전 컬럼 개수: {initial_col_count}개")
print(f"업데이트 후 최종 컬럼 개수: {final_col_count}개 (+{final_col_count - initial_col_count}개 추가됨)")
print("-" * 50)

 '기준금리', '층(고층여부)', '브랜드_Top20' 전처리 및 업데이트를 시작합니다...


C:\Users\junse\AppData\Local\Temp\ipykernel_16984\3199515914.py:16: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['기준금리'] = df['기준금리'].fillna(method='ffill').fillna(0)



 변수들이 업데이트 되었습니다

[새로 업데이트된 변수 미리보기]
    계약년  기준금리   층  고층여부  브랜드_Top20_여부
0  2021   1.0   6     0             0
1  2021   1.0   5     0             0
2  2021   1.0  12     0             0
3  2021   1.0  10     0             0
4  2021   1.0  11     0             0
--------------------------------------------------
이전 컬럼 개수: 114개
업데이트 후 최종 컬럼 개수: 118개 (+4개 추가됨)
--------------------------------------------------


In [6]:
import time
print(" 고도 데이터 추출을 시작합니다...")

# 1. 파일 불러오기
df = pd.read_csv('final_dataset.csv', low_memory=False)
initial_col_count = len(df.columns)

# 2. 고도 데이터 추출 함수
def get_elevation_batch(lat_list, lon_list):
    locations = "|".join([f"{lat},{lon}" for lat, lon in zip(lat_list, lon_list)])
    url = f"https://api.opentopodata.org/v1/srtm30m?locations={locations}"
    
    try:
        res = requests.get(url, timeout=15)
        res.raise_for_status()
        data = res.json()
        
        if "results" in data:
            return [r["elevation"] for r in data["results"]]
        else:
            return [0] * len(lat_list)
            
    except Exception as e:
        print(f"\n API 통신 에러 (0으로 임시 대체): {e}")
        return [0] * len(lat_list)

# 3. 아파트 위경도 바탕으로 고도 조회 및 추가
if '아파트위도' in df.columns and '아파트경도' in df.columns:
    elevations = []
    batch_size = 100
    total_rows = len(df)
    
    print(f"총 {total_rows}개 아파트 좌표의 고도를 조회합니다.")
    
    for i in range(0, total_rows, batch_size):
        batch_lat = df["아파트위도"].iloc[i:i+batch_size].tolist()
        batch_lon = df["아파트경도"].iloc[i:i+batch_size].tolist()
        
        elevations.extend(get_elevation_batch(batch_lat, batch_lon))
        
        time.sleep(1.2) 
        
        if (i + batch_size) % 1000 == 0 or (i + batch_size) >= total_rows:
            print(f"▶ {min(i + batch_size, total_rows)} / {total_rows} 개 고도 조회 완료...")
            
    df['고도'] = np.round(elevations, 1)

# 4. 덮어쓰기 저장 및 결과 출력
df.to_csv('final_dataset.csv', index=False, encoding='utf-8-sig')

print("\n 고도 데이터 추가가 끝났습니다")

added_cols = ['아파트위도', '아파트경도', '고도']
available_cols = [c for c in added_cols if c in df.columns]
print(df[available_cols].head())

final_col_count = len(df.columns)
print("-" * 50)
print(f"이전 컬럼 개수: {initial_col_count}개")
print(f"업데이트 후 최종 컬럼 개수: {final_col_count}개 (+{final_col_count - initial_col_count}개 추가됨)")
print("-" * 50)

 고도 데이터 추출을 시작합니다...
총 12454개 아파트 좌표의 고도를 조회합니다. (100개씩 빠르게 처리됩니다!)
▶ 1000 / 12454 개 고도 조회 완료...
▶ 2000 / 12454 개 고도 조회 완료...
▶ 3000 / 12454 개 고도 조회 완료...
▶ 4000 / 12454 개 고도 조회 완료...
▶ 5000 / 12454 개 고도 조회 완료...
▶ 6000 / 12454 개 고도 조회 완료...
▶ 7000 / 12454 개 고도 조회 완료...
▶ 8000 / 12454 개 고도 조회 완료...
▶ 9000 / 12454 개 고도 조회 완료...
▶ 10000 / 12454 개 고도 조회 완료...

 API 통신 에러 (0으로 임시 대체): 500 Server Error: Internal Server Error for url: https://api.opentopodata.org/v1/srtm30m?locations=37.6184876142783,127.007297911588%7C37.6144302680147,127.007547825596%7C37.5994822164406,127.023373194038%7C37.6007329051266,127.012568720788%7C37.592389569798,127.010468499624%7C37.5992882913524,127.038595496935%7C37.6009876756256,127.030155293619%7C37.5992448373509,127.031050089448%7C37.5992448373509,127.031050089448%7C37.6017705330128,127.03478840074%7C37.6060457872052,127.063614432719%7C37.6075574804896,127.01794878542%7C37.5941783218836,127.010321322145%7C37.6003766616613,127.022750193302%7C37.6075574804896

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import math

print("아파트 주변 4방향 고도 조회 및 경사도 계산을 시작합니다...")

# 1. 파일 불러오기
df = pd.read_csv('final_dataset.csv', low_memory=False)

# 2. 거리 및 위경도 오프셋 설정 (111m 기준)
LAT_OFFSET = 0.001
LON_OFFSET = 0.00125
DISTANCE_M = 111.0

def get_elevations(locations_str):
    url = f"https://api.opentopodata.org/v1/srtm30m?locations={locations_str}"
    try:
        res = requests.get(url, timeout=15)
        res.raise_for_status()
        data = res.json()
        if "results" in data:
            return [r["elevation"] for r in data["results"]]
    except Exception as e:
        print(f"API 통신 에러: {e}")
    return None

slopes = []

if '아파트위도' in df.columns and '아파트경도' in df.columns:
    total_rows = len(df)
    batch_size = 20
    
    print(f"총 {total_rows}개 아파트의 경사도를 계산합니다.")
    
    for i in range(0, total_rows, batch_size):
        batch_df = df.iloc[i:i+batch_size]
        locations_list = []
        
        for _, row in batch_df.iterrows():
            lat = row['아파트위도']
            lon = row['아파트경도']
            
            locations_list.extend([
                f"{lat + LAT_OFFSET},{lon}",
                f"{lat - LAT_OFFSET},{lon}",
                f"{lat},{lon + LON_OFFSET}",
                f"{lat},{lon - LON_OFFSET}"
            ])
            
        locations_str = "|".join(locations_list)
        elevations = get_elevations(locations_str)
        
        if elevations and len(elevations) == len(batch_df) * 4:
            for j in range(len(batch_df)):
                n_elev = elevations[j*4]
                s_elev = elevations[j*4 + 1]
                e_elev = elevations[j*4 + 2]
                w_elev = elevations[j*4 + 3]

                du = (e_elev - w_elev) / (2 * DISTANCE_M)

                dv = (n_elev - s_elev) / (2 * DISTANCE_M)
       
                gradient_magnitude = math.sqrt(du**2 + dv**2)
           
                slope_angle = math.degrees(math.atan(gradient_magnitude))
                slopes.append(round(slope_angle, 2))
        else:
            slopes.extend([0.0] * len(batch_df))
            
        time.sleep(1.2)
        
        if (i + batch_size) % 100 == 0 or (i + batch_size) >= total_rows:
            print(f" {min(i + batch_size, total_rows)} / {total_rows} 개 아파트 경사도 계산 완료...")

    df['경사도'] = slopes
    df.to_csv('final_dataset.csv', index=False, encoding='utf-8-sig')
    
    print("\n벡터 기반 경사도 계산 완료")
    print(df[['단지명', '경사도']].head())


아파트 주변 4방향 고도 조회 및 경사도 계산을 시작합니다...
총 12454개 아파트의 경사도를 계산합니다.
▶ 100 / 12454 개 아파트 경사도 계산 완료...
▶ 200 / 12454 개 아파트 경사도 계산 완료...
